# Authority directions — extract on `authority/output/authority_pairs_base.jsonl`

Uses the dataset built by `build_authority_pairs.ipynb` (template `"{figure} says {claim}."`).
Each record is wrapped as a single user turn via the model's chat template and the residual
stream is read at the last input token (response_first).

What we compute, per layer:

1. **Main authority direction** — diff-of-means(`source_type=authority` − `source_type=non_authority`)
   over the full `in_domain` slice. Same construction as `mech_spoof.directions.compute_refusal_direction`
   but on the authority vs non-authority split.
2. **Per pair_type sub-directions** — same diff-of-means restricted to `in_domain`,
   `cross_domain`, `anti_authority`, `institutional`. Lets us see if anti-auth gives a
   sharper axis (max contrast) and whether institutional generalises beyond human sources
   (per `authority/extended.md`).
3. **2×2 decomposition** (per `authority/extended.md`) — split `in_domain` records by
   `claim_status ∈ {plausible, dubious}`:
     - `authority_axis` = mean(+A,+E) + mean(+A,−E) − mean(−A,+E) − mean(−A,−E)
     - `epistemic_axis` = mean(+A,+E) + mean(−A,+E) − mean(+A,−E) − mean(−A,−E)
   These should be near-orthogonal if authority and epistemic accuracy are distinct.
4. **Logistic probes** per layer (label = `source_type`) on the in-domain split for the
   headline accuracy + AUROC curve.
5. **Multi-source readout** — for each prompt in `multi_source_prompts.jsonl`, project the
   activation at the last token of each annotated source span onto the main direction.
   Reports whether the score ordering matches the hand-labelled `authority_level` gradient.

Outputs saved to `OUT_DIR`:
- `result.json`, `arrays.npz`, `manifest.json` (standard `save_result_bundle` layout)
- per-prompt CSV with probe scores at the chosen best layer
- `multi_source_scores.csv` (per source mention)
- layer-accuracy plot

Edit `MODEL_KEY` in the config cell and **Run All** on an A100 (or use a small key locally).

In [ ]:
# Clone repo (for authority/ + src/) and install in editable mode.
# Works on Colab, on a pod, or locally — local checkout is preferred if it exists.
import os, sys, importlib, site, subprocess
from pathlib import Path

def _find_repo_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / 'authority').is_dir() and (p / 'src' / 'mech_spoof').is_dir():
            return p
    colab = Path('/content/Mech_spoof')
    if colab.exists():
        return colab
    colab.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ChuloIva/Mech_spoof.git', str(colab)], check=True)
    return colab

REPO_DIR = _find_repo_root()
os.environ['MECH_SPOOF_ROOT'] = str(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    subprocess.run(['pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
try:
    site.main()
except Exception:
    pass
import mech_spoof
print('mech_spoof:', mech_spoof.__file__)

In [ ]:
# Auth + Drive (Colab only).
import os
IS_COLAB = 'google.colab' in __import__('sys').modules
DRIVE_ROOT = None
if IS_COLAB:
    from google.colab import drive, userdata
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/mech_spoof_results'
print('DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY     = 'qwen3_4b'            # qwen | qwen3_4b | llama3 | llama33_70b | mistral | gemma | gemma3_4b | phi3
BATCH_SIZE    = 16                    # forward-pass batch; A100 bf16 7-9B: 16-32. T4: 4-8.
MAX_LENGTH    = 256                   # prompts are short (single sentence) — 256 is plenty
MAX_PAIRS     = None                  # None = all 7,620 pairs; set an int for quick smoke tests
CACHE_ACTS    = True                  # disk-cache per-prompt activations (resumable)

# Output location
if DRIVE_ROOT is not None:
    OUT_ROOT = Path(DRIVE_ROOT)
else:
    OUT_ROOT = Path(REPO_DIR) / 'exp_authority_directions'
slug = MODEL_CONFIGS[MODEL_KEY].slug
OUT_DIR = OUT_ROOT / slug / 'exp_authority_directions'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

# Dataset paths
AUTH_DIR    = Path(REPO_DIR) / 'authority'
PAIRS_JSONL = AUTH_DIR / 'output' / 'authority_pairs_base.jsonl'
MULTI_JSONL = AUTH_DIR / 'output' / 'multi_source_prompts.jsonl'
assert PAIRS_JSONL.exists(), f'missing {PAIRS_JSONL} — run build_authority_pairs.ipynb first'
print('PAIRS_JSONL =', PAIRS_JSONL)
print('MULTI_JSONL =', MULTI_JSONL, '(present)' if MULTI_JSONL.exists() else '(missing — will skip multi-source readout)')

In [ ]:
# Load + group the authority pairs.
import json
from collections import Counter, defaultdict

records = []
with open(PAIRS_JSONL) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

if MAX_PAIRS is not None:
    # Truncate by *pair_id* so we keep paired records together.
    keep_pairs = set(sorted({r['pair_id'] for r in records})[:MAX_PAIRS])
    records = [r for r in records if r['pair_id'] in keep_pairs]

n_records = len(records)
by_pair_type = Counter(r['pair_type'] for r in records)
by_source_type = Counter(r['source_type'] for r in records)
by_claim_status = Counter(r.get('claim_status') for r in records)
print(f'records: {n_records}')
print('  pair_type    :', dict(by_pair_type))
print('  source_type  :', dict(by_source_type))
print('  claim_status :', dict(by_claim_status))

In [ ]:
# Load model.
from mech_spoof.models import load_model
loaded = load_model(MODEL_KEY)
print(f'[{MODEL_KEY}] n_layers={loaded.n_layers} d_model={loaded.d_model} device={loaded.device}')

In [ ]:
# Wrap every record's text as a single-turn user message via the chat template.
# Residual is read at the last input token (response_first), which is the canonical
# position the rest of the codebase uses for probe training.
from mech_spoof.utils import set_seed, timer
set_seed(42)

with timer('build prompt bundles'):
    bundles = [
        loaded.template.make_user_prompt(r['text'], user_followup='')
        for r in records
    ]
print('built', len(bundles), 'bundles; example text:')
print(repr(bundles[0].text[:240]))

In [ ]:
# Extract residual at last token, batched, cache to disk (resumable).
import numpy as np
from mech_spoof.activations import extract_at_last_token_batched

cache_dir = OUT_DIR / 'act_cache' if CACHE_ACTS else None
with timer(f'extract activations (bs={BATCH_SIZE})'):
    acts = extract_at_last_token_batched(
        loaded, bundles,
        batch_size=BATCH_SIZE, max_length=MAX_LENGTH, cache_dir=cache_dir,
    )  # (n, n_layers, d_model)
print('acts:', acts.shape, acts.dtype)

In [ ]:
# Build index masks for each slice.
import numpy as np
idx = np.arange(len(records))
pair_type   = np.array([r['pair_type'] for r in records])
source_type = np.array([r['source_type'] for r in records])
claim_stat  = np.array([r.get('claim_status', '') for r in records])

MASK = {
    'all'                : np.ones(len(records), dtype=bool),
    'in_domain'          : pair_type == 'in_domain',
    'cross_domain'       : pair_type == 'cross_domain',
    'anti_authority'     : pair_type == 'anti_authority',
    'institutional'      : pair_type == 'institutional',
}
POS = source_type == 'authority'
NEG = ~POS  # non_authority OR anti_authority OR low-credibility institutional all collapse here

for k, m in MASK.items():
    print(f'  {k:18s}  n={int(m.sum()):5d}  +A={int((m & POS).sum()):5d}  -A={int((m & NEG).sum()):5d}')

In [ ]:
# Compute diff-of-means direction per slice, per layer.
import numpy as np

def diff_of_means_per_layer(acts_pos, acts_neg):
    """Return (directions (n_layers, d), norms (n_layers,), mean_pos, mean_neg)."""
    mp = acts_pos.mean(axis=0)   # (n_layers, d)
    mn = acts_neg.mean(axis=0)
    diff = mp - mn
    norms = np.linalg.norm(diff, axis=-1)
    unit = diff / (norms[:, None] + 1e-8)
    return unit, norms, mp, mn

directions_by_slice = {}    # slice_name -> (n_layers, d)
norms_by_slice      = {}
means_by_slice      = {}    # slice -> {'pos': (n_layers, d), 'neg': (n_layers, d)}

for slice_name, m in MASK.items():
    pos_mask = m & POS
    neg_mask = m & NEG
    if pos_mask.sum() < 2 or neg_mask.sum() < 2:
        print(f'  skip {slice_name}: too few samples')
        continue
    unit, nm, mp, mn = diff_of_means_per_layer(acts[pos_mask], acts[neg_mask])
    directions_by_slice[slice_name] = unit
    norms_by_slice[slice_name] = nm
    means_by_slice[slice_name] = {'pos': mp, 'neg': mn}
    best_l = int(np.argmax(nm))
    print(f'  {slice_name:18s}  ||Δμ|| peak: layer={best_l:3d}  ||Δμ||={nm[best_l]:.3f}')

main_dir = directions_by_slice['in_domain']  # canonical authority axis

In [ ]:
# 2x2 decomposition: authority axis vs epistemic axis (extended.md).
# Use the in_domain slice — that's where claim_status is well-defined for both source types.
import numpy as np

IS_IN = MASK['in_domain']
IS_PLAUS = claim_stat == 'plausible'
IS_DUB   = claim_stat == 'dubious'

cells = {}
for a_name, a_mask in [('+A', POS), ('-A', NEG)]:
    for e_name, e_mask in [('+E', IS_PLAUS), ('-E', IS_DUB)]:
        m = IS_IN & a_mask & e_mask
        cells[(a_name, e_name)] = acts[m].mean(axis=0) if m.sum() > 0 else None
        print(f'  cell {a_name}{e_name}  n={int(m.sum())}')

# Authority axis: marginalise over E. +A − (−A).
auth_diff = 0.5 * (cells[('+A', '+E')] + cells[('+A', '-E')]) \
          - 0.5 * (cells[('-A', '+E')] + cells[('-A', '-E')])
epi_diff  = 0.5 * (cells[('+A', '+E')] + cells[('-A', '+E')]) \
          - 0.5 * (cells[('+A', '-E')] + cells[('-A', '-E')])

auth_norms = np.linalg.norm(auth_diff, axis=-1)
epi_norms  = np.linalg.norm(epi_diff,  axis=-1)
auth_axis = auth_diff / (auth_norms[:, None] + 1e-8)
epi_axis  = epi_diff  / (epi_norms[:, None]  + 1e-8)

# Per-layer angle between the two axes
cos_AE = (auth_axis * epi_axis).sum(axis=-1)
angle_AE = np.degrees(np.arccos(np.clip(cos_AE, -1, 1)))
best_l_auth = int(np.argmax(auth_norms))
best_l_epi  = int(np.argmax(epi_norms))
print(f'\nauthority axis peak: layer={best_l_auth}  ||Δμ||={auth_norms[best_l_auth]:.3f}')
print(f'epistemic axis peak: layer={best_l_epi}  ||Δμ||={epi_norms[best_l_epi]:.3f}')
print(f'auth↔epi angle at layer {best_l_auth}: {angle_AE[best_l_auth]:.1f}° (cos={cos_AE[best_l_auth]:+.3f})')
print(f'auth↔epi angle at layer {best_l_epi}:  {angle_AE[best_l_epi]:.1f}° (cos={cos_AE[best_l_epi]:+.3f})')

In [ ]:
# Train per-layer logistic probes on the in_domain split (label = source_type).
# Also report accuracy on cross_domain and anti_authority held out, using the in_domain probe.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score

def fit_probes(acts_sub, labels, seed=42, test_size=0.25):
    n_layers = acts_sub.shape[1]
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(np.zeros(len(labels)), labels))
    out = {'probes': {}, 'acc': {}, 'auc': {}, 'dirs': {},
           'train_idx': train_idx, 'test_idx': test_idx}
    for L in range(n_layers):
        X = acts_sub[:, L, :]
        X = X / (np.linalg.norm(X, axis=-1, keepdims=True) + 1e-8)
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = labels[train_idx], labels[test_idx]
        clf = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=seed)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:, 1]
        out['probes'][L] = clf
        out['acc'][L] = float(accuracy_score(yte, (p > 0.5).astype(int)))
        try:
            out['auc'][L] = float(roc_auc_score(yte, p))
        except ValueError:
            out['auc'][L] = float('nan')
        w = clf.coef_[0]
        out['dirs'][L] = w / (np.linalg.norm(w) + 1e-8)
    return out

labels_all = POS.astype(int)
probe_results = {}
for slice_name in ['in_domain', 'cross_domain', 'anti_authority', 'institutional']:
    m = MASK[slice_name]
    if m.sum() < 20 or POS[m].sum() < 2 or (~POS[m]).sum() < 2:
        continue
    with timer(f'probe @ {slice_name}'):
        probe_results[slice_name] = fit_probes(acts[m], labels_all[m])
    best_l = max(probe_results[slice_name]['acc'], key=probe_results[slice_name]['acc'].get)
    print(f'  {slice_name:18s}  best layer={best_l:3d}  acc={probe_results[slice_name]["acc"][best_l]:.4f}  auc={probe_results[slice_name]["auc"][best_l]:.4f}')

# Held-out generalisation: train probe on in_domain, score on the others.
def cross_eval(train_probes, train_dirs, eval_mask, eval_label):
    n_layers = acts.shape[1]
    accs = {}; aucs = {}
    Xfull = acts[eval_mask]; yfull = labels_all[eval_mask]
    if len(yfull) == 0 or len(np.unique(yfull)) < 2:
        return None
    for L in range(n_layers):
        clf = train_probes[L]
        X = Xfull[:, L, :]; X = X / (np.linalg.norm(X, axis=-1, keepdims=True) + 1e-8)
        p = clf.predict_proba(X)[:, 1]
        accs[L] = float(accuracy_score(yfull, (p > 0.5).astype(int)))
        try:
            aucs[L] = float(roc_auc_score(yfull, p))
        except ValueError:
            aucs[L] = float('nan')
    return {'acc': accs, 'auc': aucs}

cross_results = {}
if 'in_domain' in probe_results:
    base_probes = probe_results['in_domain']['probes']
    base_dirs   = probe_results['in_domain']['dirs']
    for slice_name in ['cross_domain', 'anti_authority', 'institutional']:
        m = MASK[slice_name]
        r = cross_eval(base_probes, base_dirs, m, labels_all)
        if r is not None:
            cross_results[slice_name] = r
            best_l = max(r['acc'], key=r['acc'].get)
            print(f'  in_domain→{slice_name:14s}  best layer={best_l:3d}  acc={r["acc"][best_l]:.4f}  auc={r["auc"][best_l]:.4f}')

In [ ]:
# Cosine agreement between slices' diff-in-means directions and the in_domain probe direction.
import numpy as np

def cos_per_layer(A, B):
    nA = A / (np.linalg.norm(A, axis=-1, keepdims=True) + 1e-8)
    nB = B / (np.linalg.norm(B, axis=-1, keepdims=True) + 1e-8)
    return (nA * nB).sum(axis=-1)

print('cosine vs in_domain DIM direction (per layer summary at peak):')
for slice_name, D in directions_by_slice.items():
    c = cos_per_layer(D, directions_by_slice['in_domain'])
    peak_layer = int(np.argmax(np.abs(c)))
    print(f'  {slice_name:18s}  peak |cos| layer={peak_layer:3d}  cos={c[peak_layer]:+.3f}  (mid-layer cos={c[len(c)//2]:+.3f})')

if 'in_domain' in probe_results:
    probe_D = np.stack([probe_results['in_domain']['dirs'][L] for L in range(acts.shape[1])], axis=0)
    c = cos_per_layer(probe_D, directions_by_slice['in_domain'])
    print('\nprobe vs DIM (in_domain): mid-layer cos=', float(c[len(c)//2]))

In [ ]:
# Plot per-layer accuracy (logistic probe) for each slice, + cross-domain generalisation.
import matplotlib.pyplot as plt
import numpy as np

n_layers = acts.shape[1]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
for slice_name, res in probe_results.items():
    ys = [res['acc'][L] for L in range(n_layers)]
    ax.plot(ys, marker='.', label=f'{slice_name} (n={int(MASK[slice_name].sum())})')
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.set_xlabel('layer'); ax.set_ylabel('test accuracy')
ax.set_title(f'{MODEL_KEY}: logistic probe (authority vs non-authority)')
ax.legend(fontsize=8); ax.set_ylim(0.4, 1.02)

ax = axes[1]
if 'in_domain' in probe_results:
    base_acc = [probe_results['in_domain']['acc'][L] for L in range(n_layers)]
    ax.plot(base_acc, marker='.', label='in_domain (train)')
for slice_name, r in cross_results.items():
    ys = [r['acc'][L] for L in range(n_layers)]
    ax.plot(ys, marker='.', linestyle='--', label=f'in_domain→{slice_name}')
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.set_xlabel('layer'); ax.set_ylabel('test accuracy')
ax.set_title(f'{MODEL_KEY}: generalisation of in_domain probe')
ax.legend(fontsize=8); ax.set_ylim(0.4, 1.02)

plt.tight_layout()
plt.savefig(OUT_DIR / 'layer_accuracy.png', dpi=130)
plt.show()

In [ ]:
# Multi-source readout: project residual at the last token of each annotated source
# span onto the in_domain authority direction, layer by layer.
import numpy as np, json, csv
from mech_spoof.activations import extract_residual_stream

multi_rows = []
if MULTI_JSONL.exists():
    prompts = [json.loads(l) for l in open(MULTI_JSONL) if l.strip()]
    if MAX_PAIRS is not None:
        prompts = prompts[:max(4, MAX_PAIRS // 200)]
    print(f'multi-source prompts: {len(prompts)}')
    AUTH_LEVEL = {'high': 3, 'mid': 2, 'low': 1, 'anti': 0}
    # Pick the best in_domain layer.
    if 'in_domain' in probe_results:
        best_l = max(probe_results['in_domain']['acc'], key=probe_results['in_domain']['acc'].get)
    else:
        best_l = int(np.argmax(norms_by_slice['in_domain']))
    dir_vec = directions_by_slice['in_domain'][best_l]
    dir_vec = dir_vec / (np.linalg.norm(dir_vec) + 1e-8)
    print(f'  using in_domain direction at layer {best_l}')

    for p in prompts:
        text = p['prompt']
        # Wrap as a single user message; same tokenizer/chat template as everywhere else.
        b = loaded.template.make_user_prompt(text, user_followup='')
        # Tokenize the chat-templated TEXT to recover offset mappings into the templated string.
        enc = loaded.tokenizer(b.text, return_offsets_mapping=True, add_special_tokens=False)
        ids = enc['input_ids']
        offsets = enc['offset_mapping']
        # Reconcile possible BOS delta vs the canonical bundle ids
        delta = len(b.input_ids) - len(ids)  # if positive, BOS was prepended
        # Run a single forward pass and capture the full residual stream.
        full = extract_residual_stream(loaded, b.input_ids)   # (n_layers, seq_len, d) cpu fp32
        for src in p['sources']:
            needle = src['text']
            char_start = b.text.rfind(needle)
            if char_start < 0:
                continue
            char_end = char_start + len(needle)
            # Find last token whose offset overlaps the needle.
            last_tok = None
            for ti, (a, c) in enumerate(offsets):
                if a == c:
                    continue
                if a < char_end and c > char_start:
                    last_tok = ti
            if last_tok is None:
                continue
            pos = last_tok + delta
            v = full[best_l, pos].numpy()
            vn = v / (np.linalg.norm(v) + 1e-8)
            score = float(vn @ dir_vec)
            multi_rows.append({
                'prompt_id': p['id'], 'domain': p.get('domain', ''),
                'source_text': needle, 'authority_level': src['authority_level'],
                'authority_level_num': AUTH_LEVEL.get(src['authority_level'], -1),
                'layer': best_l, 'token_pos': pos,
                'projection': score,
            })

    if multi_rows:
        # Per-prompt Spearman-ish: just check that mean(high) > mean(mid) > mean(low) > mean(anti).
        buckets = {}
        for r in multi_rows:
            buckets.setdefault(r['authority_level'], []).append(r['projection'])
        print('\nmean projection on authority direction by authority_level:')
        for lv in ('high', 'mid', 'low', 'anti'):
            if lv in buckets:
                arr = np.array(buckets[lv])
                print(f'  {lv:5s}  n={len(arr):4d}  mean={arr.mean():+.4f}  std={arr.std():+.4f}')

        out_csv = OUT_DIR / 'multi_source_scores.csv'
        with open(out_csv, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(multi_rows[0].keys()))
            w.writeheader(); w.writerows(multi_rows)
        print('saved', out_csv)
else:
    print('no multi_source_prompts.jsonl — skipped')

In [ ]:
# Save result bundle (directions, probes, accuracies, metadata).
import numpy as np, json, pickle
from mech_spoof.io import save_result_bundle

n_layers = acts.shape[1]
arrays = {}
for slice_name, D in directions_by_slice.items():
    arrays[f'dim_dir__{slice_name}'] = D.astype(np.float32)
    arrays[f'dim_norms__{slice_name}'] = norms_by_slice[slice_name].astype(np.float32)
arrays['authority_axis_2x2'] = auth_axis.astype(np.float32)
arrays['epistemic_axis_2x2'] = epi_axis.astype(np.float32)
arrays['authority_axis_norms_2x2'] = auth_norms.astype(np.float32)
arrays['epistemic_axis_norms_2x2'] = epi_norms.astype(np.float32)
arrays['auth_epi_cos_2x2'] = cos_AE.astype(np.float32)

probe_dirs_npz = {}
for slice_name, res in probe_results.items():
    D = np.stack([res['dirs'][L] for L in range(n_layers)], axis=0)
    arrays[f'probe_dir__{slice_name}'] = D.astype(np.float32)
    arrays[f'probe_acc__{slice_name}'] = np.array([res['acc'][L] for L in range(n_layers)], dtype=np.float32)
    arrays[f'probe_auc__{slice_name}'] = np.array([res['auc'][L] for L in range(n_layers)], dtype=np.float32)
for slice_name, r in cross_results.items():
    arrays[f'probe_acc__in_domain_to_{slice_name}'] = np.array([r['acc'][L] for L in range(n_layers)], dtype=np.float32)
    arrays[f'probe_auc__in_domain_to_{slice_name}'] = np.array([r['auc'][L] for L in range(n_layers)], dtype=np.float32)

result_json = {
    'model_key': MODEL_KEY,
    'hf_id': loaded.cfg.hf_id,
    'n_layers': int(n_layers),
    'd_model': int(acts.shape[2]),
    'n_records': int(len(records)),
    'pair_type_counts': {k: int(v) for k, v in by_pair_type.items()},
    'source_type_counts': {k: int(v) for k, v in by_source_type.items()},
    'claim_status_counts': {str(k): int(v) for k, v in by_claim_status.items()},
    'directions_slices': list(directions_by_slice.keys()),
    'probe_slices': list(probe_results.keys()),
    'cross_eval_slices': list(cross_results.keys()),
    'best_layer_per_slice': {
        s: int(max(res['acc'], key=res['acc'].get)) for s, res in probe_results.items()
    },
    'best_acc_per_slice': {
        s: float(res['acc'][max(res['acc'], key=res['acc'].get)]) for s, res in probe_results.items()
    },
    '2x2_authority_peak_layer': int(np.argmax(auth_norms)),
    '2x2_epistemic_peak_layer': int(np.argmax(epi_norms)),
    '2x2_auth_epi_cos_at_auth_peak': float(cos_AE[int(np.argmax(auth_norms))]),
}

pickles = {
    f'probes__{s}': res['probes'] for s, res in probe_results.items()
}

save_result_bundle(
    OUT_DIR,
    json_obj=result_json,
    arrays=arrays,
    pickles=pickles,
    manifest_extras={'experiment': 'exp_authority_directions', 'model_key': MODEL_KEY},
)
print('saved bundle to', OUT_DIR)
print(json.dumps({k: v for k, v in result_json.items() if k != 'pair_type_counts'}, indent=2))

In [ ]:
# Free model + summary.
from mech_spoof.models import free_model
free_model(loaded)
print('done.')